In [ ]:
import os
import sys

# 実行環境の確認
if "google.colab" in sys.modules:
    print("✓ Running in Google Colab")
else:
    print("✗ Running locally (not Colab)")

print(f"Python version: {sys.version}")
print(f"Executable: {sys.executable}")
print(f"Working Directory: {os.getcwd()}")

In [ ]:
# Colab で実行している場合、リポジトリをクローンする
#!git clone -b master https://github.com/oreilly-japan/deep-learning-from-scratch-2.git
#%cd deep-learning-from-scratch-2
#sys.path.append('.')

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 共通レイヤを管理する common ディレクトリからインポートする想定
#from common.time_layers import *
#from common.base_model import BaseModel
#from common.optimizer import SGD
#from common.trainer import RnnlmTrainer
#from common.util import eval_perplexity
#from dataset import ptb

# 第 6 章 ゲート付き RNN

## 6.1 RNN の問題点
前章で学んだシンプルな RNN は、構造が単純で実装も容易ですが、大きな欠点がある  
それは、「時系列データの長期の依存関係を学習するのが苦手」という点である  

### 6.1.1 RNN の復習
RNN は、前時刻の隠れ状態 $h_{t-1}$ と現時刻の入力 $x_t$ を受け取り、新しい隠れ状態 $h_t$ を出力する  

#### RNN レイヤの構造

p224 の図 6-1 を参照

RNN レイヤ内で行われる計算は以下の通りである：  
$$h_t = \tanh(h_{t-1}W_h + x_tW_x + b)$$

### 6.1.2 勾配消失もしくは勾配爆発
RNN が長期記憶を苦手とする理由は、BPTT (Backpropagation Through Time) において勾配消失 (vanishing gradients) または勾配爆発 (exploding gradients) が起こるためである  

例えば、「Tom was watching TV in his room. Mary came into the room. Mary said hi to [?]」という文章で、正解の「Tom」を予測するには、かなり過去の情報である「Tom」を記憶しておく必要がある。しかし、逆伝播の過程で勾配が途中で弱まると、重みパラメータが更新されず、長期の依存関係を学習できなくなる  

### 6.1.3 勾配消失・爆発の原因
勾配が変化する要因は、主に以下の2つの演算にある：  

1. tanh による影響：  
    $\tanh(x)$ の微分は $1 - y^2$ であり、その値は常に 1.0 以下である。逆伝播で $\tanh$ ノードを通るたびに、勾配は小さくなっている  

2. MatMul (行列の積) による影響：  
    同じ重み $W_h$ を時間方向の数だけ掛け合わせるため、重みの値が 1 より大きければ指数的に増加 (爆発) し、1 より小さければ指数的に減少 (消失) する  

### 勾配爆発のシミュレーション
教科書にある実験を再現し、重み $W_h$ の値によって勾配がどのように変化するかを可視化する  

In [ ]:
N = 2   # ミニバッチサイズ
H = 3   # 隠れ状態ベクトルの次元数
T = 20  # 時系列データの長さ

dh = np.ones((N, H))
np.random.seed(3)
Wh = np.random.randn(H, H) * 1.0  # 重みの初期値 (標準偏差を調整して実験)

norm_list = []
for t in range(T):
    dh = np.dot(dh, Wh.T)
    norm = np.sqrt(np.sum(dh**2)) / N
    norm_list.append(norm)

# 可視化
plt.plot(np.arange(len(norm_list)), norm_list)
plt.xticks()
plt.xlabel('time step')
plt.ylabel('norm')
plt.title('Gradient Exploding Simulation')
plt.show()

### 固有値による勾配の挙動変化
行列 $W_h$ を繰り返し掛ける際の勾配の挙動は、その行列の「最大固有値」に依存する。固有値が 1 より大きいか小さいかで、爆発か消失かが決まる様子を確認してみる  

In [ ]:
def simulate_gradient(scale):
    Wh = np.random.randn(H, H) * scale
    # 固有値を計算
    eigenvalues = np.linalg.eigvals(Wh)
    max_ev = np.max(np.abs(eigenvalues))

    dh = np.ones((N, H))
    norms = []
    for t in range(50):
        dh = np.dot(dh, Wh.T)
        norms.append(np.sqrt(np.sum(dh**2)) / N)
    return norms, max_ev

In [ ]:
scales = [0.4, 0.6]  # 消失と爆発の境界付近
plt.figure(figsize=(10, 5))
for s in scales:
    norms, ev = simulate_gradient(s)
    plt.plot(norms, label=f'Max Eigenvalue: {ev:.2f}')

plt.yscale('log')
plt.xlabel('time step')
plt.ylabel('norm (log scale)')
plt.legend()
plt.title('Vanishing vs Exploding Gradients (Eigenvalue focus)')
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.show()

### 6.1.4 勾配爆発への対策：勾配クリッピング
勾配爆発を防ぐための伝統的な手法として、勾配クリッピング (gradients clipping) がある。これは、勾配のノルムがしきい値を超えた場合に、勾配を修正するシンプルな手法である  

In [ ]:
def clip_grads(grads, max_norm):
    total_norm = 0
    for grad in grads:
        total_norm += np.sum(grad ** 2)
    total_norm = np.sqrt(total_norm)

    rate = max_norm / (total_norm + 1e-6)
    if rate < 1:
        for grad in grads:
            grad *= rate

In [ ]:
# 動作確認用のダミー勾配
dW1 = np.random.rand(3, 3) * 10
dW2 = np.random.rand(3, 3) * 10
grads = [dW1, dW2]
max_norm = 5.0

print(f"Before clipping norm: {np.sqrt(np.sum(dW1**2) + np.sum(dW2**2)):.2f}")
clip_grads(grads, max_norm)
print(f"After clipping norm: {np.sqrt(np.sum(dW1**2) + np.sum(dW2**2)):.2f}")

# 6.2 勾配消失と LSTM

シンプルな RNN が抱える勾配消失の問題を解決するために、アーキテクチャを根本から見直したものを「ゲート付きRNN」という  
この章では、その代表格である LSTM (Long Short-Term Memory) の仕組みを学ぶ  

## 6.2.1 LSTM のインタフェース
LSTM と RNN の最大の違いは、隠れ状態 $h$ に加えて、記憶セル $c$ という専用の経路を持つことにある  

- 記憶セル $c_t$：  
    LSTM レイヤ内だけで完結する記憶部であり、他のレイヤへは出力されない  

- 隠れ状態 $h_t$：  
    RNN と同様に、他のレイヤ (上方向) や次時刻のレイヤへ出力される  

### LSTM　レイヤの入出力インタフェース

p235 の図 6-11 を参照

記憶セル $c_t$ は、過去から現在までの必要な情報がすべて格納されるように学習される

## 6.2.2 ゲートの仕組み
LSTM では、情報の流れをコントロールするために「ゲート (gate)」という仕組みを用いる。ゲートは、データの流れる量を 0.0〜1.0 の範囲で調整する (1.0は全開、0.0は遮断)。この「開き具合」自体も、学習によってデータから自動的に決定される  

- シグモイド関数 ($\sigma$)：  
    ゲートの開き具合を求めるために使用される  

- tanh 関数：  
    実質的な「情報」の強弱 (-1.0 〜 1.0) を表すために使用される  

## 6.2.3 各ゲートの詳細と計算式

LSTM は 3 つのゲートと、1 つの新しい記憶候補を用いて計算を行う  

### 1. forget ゲート (忘却ゲート)
「何を忘れるか」を制御する  
前時刻の記憶 $c_{t-1}$ から不要な情報を削除する  
$$f = \sigma(x_t W_x^{(f)} + h_{t-1} W_h^{(f)} + b^{(f)})$$

### 2. input ゲート (入力ゲート)
「何を覚えるか」を制御する  
新しく追加する情報の価値を判断する  
$$i = \sigma(x_t W_x^{(i)} + h_{t-1} W_h^{(i)} + b^{(i)})$$

### 3. 新しい記憶セル (候補)
記憶セルに新しく追加するための「情報」そのもの
$$g = \tanh(x_t W_x^{(g)} + h_{t-1} W_h^{(g)} + b^{(g)})$$

### 4. output ゲート (出力ゲート)
「何を出力するか」を制御する  
更新された記憶セル $c_t$ のうち、次時刻の隠れ状態として重要なものを調整する  
$$o = \sigma(x_t W_x^{(o)} + h_{t-1} W_h^{(o)} + b^{(o)})$$

### 5. 状態の更新式
最終的な $c_t$ と $h_t$ は、これらの値を組み合わせて求められる  

- $c_t = f \odot c_{t-1} + g \odot i$ (忘却と新規追加)  

- $h_t = o \odot \tanh(c_t)$ (出力の調整)  

$\odot$ はアダマール積 (要素ごとの積) を表す  

### LSTM 内部の計算構造

p242 の図 6-18 を参照

## 6.2.7 LSTM の勾配の流れ：なぜ勾配消失が起きないのか
LSTM が長期記憶を保持できる理由は、記憶セル $c$ の逆伝播にある

1. 「＋」ノード：  
    勾配をそのまま流すため、劣化が起きない  

2. 「×」ノード：  
    RNN のような「行列の積」ではなく、「要素ごとの積 (アダマール積)」  

3. 忘却ゲートの制御：  
    forget ゲートが「忘れてはいけない」と判断した要素には、勾配が劣化することなく過去へ伝わる  

この仕組みにより、重要な情報は長い時間を越えて伝播することが可能になる  

### シグモイド関数による「ゲート」のシミュレーション
本書の解説を補足するために、ゲートがどのように情報を遮断・通過させるかを視覚的に確認する実験を追加する

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [ ]:
# 1. 入力データ（何らかの情報）
data = np.sin(np.linspace(0, 10, 100))

# 2. ゲートの制御信号（途中で忘却し、また思い出すような動き）
# 前半は通過(1.0)、中間は遮断(0.0)、後半は半分通過(0.5)をシミュレート
gate_input = np.array([5.0]*30 + [-5.0]*40 + [0.0]*30)
gate_values = sigmoid(gate_input)

# 3. ゲートを通過した後のデータ
output = data * gate_values

In [ ]:
# 可視化
plt.figure(figsize=(12, 6))
plt.subplot(3, 1, 1)
plt.plot(data, color='blue')
plt.title("Original Information (Candidate g)")
plt.grid(True)

plt.subplot(3, 1, 2)
plt.step(range(100), gate_values, color='red', where='post')
plt.title("Gate Opening Rate (Sigmoid Output)")
plt.ylim(-0.1, 1.1)
plt.grid(True)

plt.subplot(3, 1, 3)
plt.plot(output, color='green', linewidth=2)
plt.title("Filtered Information (Data * Gate)")
plt.grid(True)

plt.tight_layout()
plt.show()

### この節のまとめ
- 記憶セル $c$ は情報の「保管庫」であり、逆伝播で勾配が消えにくい構造になっている  

- 3 つのゲート (forget, input, output) が、情報の忘却、取り込み、出力を動的にコントロールする  

- アダマール積を用いることで、時刻ごとに異なるゲート値が勾配の強さを調整し、長期依存関係の学習を可能にする  

# 6.3 LSTM の実装

ここでは、これまでに学んだ LSTM の計算を Python のクラスとして実装する  

### 効率化のポイント：4 つのアフィン変換をまとめる
LSTM で行う 4 つのゲート ($f, g, i, o$) の計算は、いずれも $x_t W_x + h_{t-1} W_h + b$ という同じ形式のアフィン変換である。これらを個別に計算するのではなく、4 つの重みを 1 つの大きな行列にまとめることで、行列計算を一括して行い、計算を高速化する  

#### 重みをまとめた LSTM の計算グラフ

p245 の図 6-20 を参照

## 6.3.1 LSTM クラスの実装
単一の時刻を処理するクラスである。`slice` ノードを用いて一括計算された結果を分割する  

In [ ]:
class LSTM:
    def __init__(self, Wx, Wh, b):
        """
        Wx: 入力用の重み (D, 4H)
        Wh: 隠れ状態用の重み (H, 4H)
        b: バイアス (4H,)
        """
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.cache = None

    def forward(self, x, h_prev, c_prev):
        Wx, Wh, b = self.params
        N, H = h_prev.shape

        # 4つ分のアフィン変換を一括計算
        A = np.dot(x, Wx) + np.dot(h_prev, Wh) + b

        # sliceして各ゲートに分配
        f = A[:, :H]
        g = A[:, H:2*H]
        i = A[:, 2*H:3*H]
        o = A[:, 3*H:]

        f = sigmoid(f)
        g = np.tanh(g)
        i = sigmoid(i)
        o = sigmoid(o)

        c_next = f * c_prev + g * i
        h_next = o * np.tanh(c_next)

        self.cache = (x, h_prev, c_prev, i, f, g, o, c_next)
        return h_next, c_next

    def backward(self, dh_next, dc_next):
        Wx, Wh, b = self.params
        x, h_prev, c_prev, i, f, g, o, c_next = self.cache

        tanh_c_next = np.tanh(c_next)

        ds = dc_next + (dh_next * o) * (1 - tanh_c_next ** 2)

        dc_prev = ds * f

        df = ds * c_prev
        dg = ds * i
        di = ds * g
        do = dh_next * tanh_c_next

        # ゲートの微分 (sigmoidとtanh)
        df *= f * (1 - f)
        dg *= (1 - g ** 2)
        di *= i * (1 - i)
        do *= o * (1 - o)

        # 分割していた勾配を横に連結 (dA)
        dA = np.hstack((df, dg, di, do))

        dWh = np.dot(h_prev.T, dA)
        dWx = np.dot(x.T, dA)
        db = dA.sum(axis=0)

        self.grads[...] = dWx
        self.grads[...] = dWh
        self.grads[...] = db

        dx = np.dot(dA, Wx.T)
        dh_prev = np.dot(dA, Wh.T)

        return dx, dh_prev, dc_prev

## 6.3.2 TimeLSTM クラスの実装
$T$ ステップ分の時系列データをまとめて処理するレイヤ

In [ ]:
class TimeLSTM:
    def __init__(self, Wx, Wh, b, stateful=False):
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.layers = None

        self.h, self.c = None, None
        self.dh = None
        self.stateful = stateful

    def forward(self, xs):
        Wx, Wh, b = self.params
        N, T, D = xs.shape
        H = Wh.shape[0]

        self.layers = []
        hs = np.empty((N, T, H), dtype='f')

        if not self.stateful or self.h is None:
            self.h = np.zeros((N, H), dtype='f')
        if not self.stateful or self.c is None:
            self.c = np.zeros((N, H), dtype='f')

        for t in range(T):
            layer = LSTM(*self.params)
            self.h, self.c = layer.forward(xs[:, t, :], self.h, self.c)
            hs[:, t, :] = self.h

            self.layers.append(layer)

        return hs

    def backward(self, dhs):
        Wx, Wh, b = self.params
        N, T, H = dhs.shape
        D = Wx.shape[0]

        dxs = np.empty((N, T, D), dtype='f')
        dh, dc = 0, 0

        grads = [0, 0, 0]
        for t in reversed(range(T)):
            layer = self.layers[t]
            dx, dh, dc = layer.backward(dhs[:, t, :] + dh, dc)
            dxs[:, t, :] = dx
            for i, grad in enumerate(layer.grads):
                grads[i] += grad

        for i, grad in enumerate(grads):
            self.grads[i][...] = grad
        self.dh = dh
        return dxs

    def set_state(self, h, c=None):
        self.h, self.c = h, c

    def reset_state(self):
        self.h, self.c = None, None

### LSTM 内部のゲート挙動の可視化
特定の入力に対して、LSTM の「忘却ゲート」や「出力ゲート」がどのように反応するかを、疑似データを使って可視化する  

In [ ]:
# ハイパーパラメータの設定
N, T, D, H = 1, 50, 10, 5
Wx = (np.random.randn(D, 4*H) / np.sqrt(D)).astype('f')
Wh = (np.random.randn(H, 4*H) / np.sqrt(H)).astype('f')
b = np.zeros(4*H).astype('f')

# シーケンシャルな入力データ作成
xs = np.random.randn(N, T, D)
# 時刻 10-15 で強い信号を入れる
xs[0, 10:15, :] += 5.0

time_lstm = TimeLSTM(Wx, Wh, b, stateful=True)
hs = time_lstm.forward(xs)

# 各時刻のゲートの平均的な「開き具合」をシミュレートして保存
# (内部のレイヤから gate 値を取り出すために少し特殊なアクセスをします)
f_values = []
o_values = []
for layer in time_lstm.layers:
    x, h_prev, c_prev, i, f, g, o, c_next = layer.cache
    f_values.append(np.mean(f))
    o_values.append(np.mean(o))

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(f_values, label='Forget Gate (f) opening rate', color='orange')
plt.plot(o_values, label='Output Gate (o) opening rate', color='blue')
plt.axvspan(10, 15, color='red', alpha=0.1, label='Strong input signal')
plt.title("Dynamics of LSTM Gates Over Time")
plt.xlabel("Time step")
plt.ylabel("Mean Gate Activation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### この節のまとめ
- 重みの統合：  
    4 つのアフィン変換を 1 つにまとめることで、行列計算のオーバーヘッドを減らし高速化を実現する  

- 記憶セルの永続性：  
    `TimeLSTM` では `h` と `c` の両方をメンバ変数として保持することで、Truncated BPTT における長い依存関係の維持を可能にしている  

- 逆伝播の連結：  
    `LSTM` の `backward` では、4 つに分かれていたゲートの微分を `np.hstack` で連結して、一括アフィン変換の勾配 `dA` を求めている  

# 6.4 LSTM を使った言語モデル

これまでに実装した `TimeLSTM` を使って、本格的な「言語モデル (RNNLM)」を構築する。第 5 章で作成した `SimpleRnnlm` との最大の違いは、RNN レイヤが LSTM レイヤに置き換わっている点である。この変更により、モデルは長期の依存関係を学習する能力を獲得する  

## 6.4.1 RNNLM の全体構造
本節で実装する `Rnnlm` クラスは、以下のレイヤ構成を持ちます。

1.  Time Embedding： 単語 ID を分散表現に変換

2.  Time LSTM： 長期記憶を保持しながら時系列データを処理

3.  Time Affine： LSTM の出力をスコア (語彙サイズ) に変換

4.  Time Softmax with Loss： スコアを確率に変換し、損失を算出

### Rnnlm のネットワーク構造

p253 の図 6-26 を参照

## 6.4.2 Rnnlm クラスの実装
第 5 章のモデルを拡張し、パラメータの保存・読み込み機能や、次章の文章生成で使う `predict()` メソッドを備えたクラスを定義する

In [ ]:
class Rnnlm(BaseModel):
    def __init__(self, vocab_size=10000, wordvec_size=100, hidden_size=100):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        # 重みの初期化
        embed_W = (rn(V, D) / 100).astype('f')
        lstm_Wx = (rn(D, 4 * H) / np.sqrt(D)).astype('f')
        lstm_Wh = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_b = np.zeros(4 * H).astype('f')
        affine_W = (rn(H, V) / np.sqrt(H)).astype('f')
        affine_b = np.zeros(V).astype('f')

        # レイヤの生成
        self.layers = [
            TimeEmbedding(embed_W),
            TimeLSTM(lstm_Wx, lstm_Wh, lstm_b, stateful=True),
            TimeAffine(affine_W, affine_b)
        ]
        self.loss_layer = TimeSoftmaxWithLoss()
        self.lstm_layer = self.layers[1]

        # すべての重みと勾配をリストにまとめる
        self.params, self.grads = [], []
        for layer in self.layers:
            self.params += layer.params
            self.grads += layer.grads

    def predict(self, xs):
        for layer in self.layers:
            xs = layer.forward(xs)
        return xs

    def forward(self, xs, ts):
        score = self.predict(xs)
        loss = self.loss_layer.forward(score, ts)
        return loss

    def backward(self, dout=1):
        dout = self.loss_layer.backward(dout)
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def reset_state(self):
        self.lstm_layer.reset_state()

## 6.4.3 RNNLM の学習と評価
PTB データセットを用いて学習を行う。ここでは勾配爆発を防ぐための勾配クリッピングを適用する  

In [ ]:
# ハイパーパラメータの設定
batch_size = 100
wordvec_size = 100
hidden_size = 100  # RNN の隠れ状態ベクトルの要素数
time_size = 35  # RNN を展開するサイズ
lr = 20.0
max_epoch = 1
max_grad = 0.25

# 学習データの読み込み
corpus, word_to_id, id_to_word = ptb.load_data('train')
corpus_test, _, _ = ptb.load_data('test')
vocab_size = len(word_to_id)
xs = corpus[:-1]
ts = corpus[1:]

# モデルの生成
model = Rnnlm(vocab_size, wordvec_size, hidden_size)
optimizer = SGD(lr)
trainer = RnnlmTrainer(model, optimizer)

# 勾配クリッピングを適用して学習
trainer.fit(xs, ts, max_epoch, batch_size, time_size, max_grad,
            eval_interval=20)
trainer.plot()

# テストデータで評価
model.reset_state()
ppl_test = eval_perplexity(model, corpus_test)
print('test perplexity: ', ppl_test)

### 学習済みモデルによる「次の単語予測」の確認
数値としてのパープレキシティだけでなく、実際にモデルがどのような予測を行っているか、特定の単語の次に来る確率が高い単語をリストアップしてみる  

In [ ]:
def predict_next_words(model, query_word, word_to_id, id_to_word, top_k=5):
    if query_word not in word_to_id:
        print(f"'{query_word}' は語彙に含まれていません。")
        return

    query_id = word_to_id[query_word]
    # 入力を (batch_size=1, time_size=1) に整形
    x = np.array([[query_id]])

    # 隠れ状態をリセットせずに予測 (直前の単語の影響を見る)
    score = model.predict(x)
    # 最後の時刻の出力を取り出し Softmax 適用
    p = softmax(score[0, -1, :])

    # 確率が高い順にインデックスを取得
    top_ids = p.argsort()[::-1][:top_k]

    print(f"\n[入力単語]: {query_word}")
    for i in top_ids:
        print(f"  - {id_to_word[i]:10s}: {p[i]:.4f}")

In [ ]:
model = Rnnlm(vocab_size, wordvec_size, hidden_size)
model.load_params('./ch06/Rnnlm.pkl')

In [ ]:
# 学習済みモデルがある場合、以下のように確認できます
predict_next_words(model, 'you', word_to_id, id_to_word)
predict_next_words(model, 'say', word_to_id, id_to_word)

### このセクションのポイント
- LSTM の威力：  
    `SimpleRnnlm` では太刀打ちできなかった大規模な PTB データセットに対しても、LSTM を用いることで学習が進むようになる  

- 勾配クリッピング：  
    学習の安定化には、`max_grad` を用いた勾配の抑制が不可欠  

- パープレキシティ：  
    言語モデルの「迷い」の度合いを数値化し、学習の進捗を明確に評価できる  

# 6.5 RNNLM のさらなる改善

6.4 節で実装した LSTM による言語モデル (RNNLM) は、シンプルな RNN に比べて大きな進歩を遂げましたが、現代の最先端の精度にはまだ及ばない。本節では、以下の 3 つの重要な改善を加え、モデルをさらに強力にする  

1.  LSTM レイヤの多層化  

2.  Dropout による過学習の抑制  

3.  重み共有 (Weight Tying)  

## 6.5.1 LSTM レイヤの多層化
複雑な依存関係を持つ時系列データを学習する場合、LSTM レイヤを深く重ねることが有効。層を増やすことで、より抽象的で複雑なパターンを学習する能力が高まる  

### 2 層の LSTM を用いた RNNLM の構造

p260 の図 6-29 を参照

## 6.5.2 Dropout による過学習の抑制
層を深くすると表現力が増す一方で、過学習 (overfitting) を起こしやすくなる。特に RNN は通常のフィードフォワードネットワークよりも過学習しやすい傾向がある  

### 挿入位置の注意点
RNN において、Dropout を「時系列方向 (時間軸)」に挿入するのは避けるべきである。時間が進むにつれてノイズが蓄積し、情報が失われてしまうからである  

効果的な挿入位置は、「深さ方向（垂直方向）」になる。これにより、時間軸の情報を壊すことなく過学習を抑制できる  

#### 深さ方向に Dropout を適用したモデル

p264 の図 6-33 を参照

## 6.5.3 重み共有 (Weight Tying)
重み共有 (Weight Tying) は、Embedding レイヤの重みと、Affine レイヤの重みを結びつける (共有する) 非常にシンプルなトリック  

- 仕組み：  
    Embedding レイヤの重み行列 (語彙数 $\times$ 隠れ状態数) の転置を、Affine レイヤの重みとして利用する  

- メリット：  
    学習すべきパラメータ数が大幅に減り、結果として学習が容易になり過学習も抑制される  


## 6.5.4 より良い RNNLM の実装
これら 3 つの改善点を取り入れた `BetterRnnlm` クラスを実装する  

In [ ]:
class BetterRnnlm:
    def __init__(self, vocab_size=10000, wordvec_size=650,
                 hidden_size=650, dropout_ratio=0.5):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        embed_W = (rn(V, D) / 100).astype('f')
        # 1層目のLSTM用
        lstm_Wx1 = (rn(D, 4 * H) / np.sqrt(D)).astype('f')
        lstm_Wh1 = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_b1 = np.zeros(4 * H).astype('f')
        # 2層目のLSTM用
        lstm_Wx2 = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_Wh2 = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_b2 = np.zeros(4 * H).astype('f')
        affine_b = np.zeros(V).astype('f')

        # 改良ポイントの実装
        self.layers = [
            TimeEmbedding(embed_W),
            TimeDropout(dropout_ratio),
            TimeLSTM(lstm_Wx1, lstm_Wh1, lstm_b1, stateful=True),
            TimeDropout(dropout_ratio),
            TimeLSTM(lstm_Wx2, lstm_Wh2, lstm_b2, stateful=True),
            TimeDropout(dropout_ratio),
            TimeAffine(embed_W.T, affine_b)  # 重みを共有
        ]
        self.loss_layer = TimeSoftmaxWithLoss()
        self.lstm_layers = [self.layers[2], self.layers[4]]
        self.drop_layers = [self.layers[1], self.layers[3], self.layers[5]]

        self.params, self.grads = [], []
        for layer in self.layers:
            self.params += layer.params
            self.grads += layer.grads

    def predict(self, xs, train_flg=False):
        for layer in self.drop_layers:
            layer.train_flg = train_flg

        for layer in self.layers:
            xs = layer.forward(xs)
        return xs

    def forward(self, xs, ts, train_flg=True):
        score = self.predict(xs, train_flg)
        loss = self.loss_layer.forward(score, ts)
        return loss

    def backward(self, dout=1):
        dout = self.loss_layer.backward(dout)
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def reset_state(self):
        for layer in self.lstm_layers:
            layer.reset_state()

## 6.5.5 学習と評価の工夫
学習をより効率的に進めるために、以下の実践的なテクニックを導入する  

- 学習係数のスケジューリング：  
    エポックごとに検証データのパープレキシティを評価し、値が悪化したときのみ学習係数を (例えば 1/4 に) 下げる  

### パラメータ数の比較
重み共有によってどれだけパラメータが削減されるかを、簡単なスクリプトで可視化してみる  

In [ ]:
def count_params(V, D, H, tied=False):
    # Embedding: V * D
    embed = V * D
    # LSTM x 2層: (D*4H + H*4H + 4H) + (H*4H + H*4H + 4H)
    lstm1 = (D * 4 * H) + (H * 4 * H) + (4 * H)
    lstm2 = (H * 4 * H) + (H * 4 * H) + (4 * H)
    # Affine: H * V + V
    affine = (H * V) + V if not tied else V # tied の場合はバイアスのみ加算

    return embed + lstm1 + lstm2 + affine

In [ ]:
V, D, H = 10000, 650, 650
params_normal = count_params(V, D, H, tied=False)
params_tied = count_params(V, D, H, tied=True)

print(f"通常モデルのパラメータ数: {params_normal:,}")
print(f"重み共有モデルのパラメータ数: {params_tied:,}")
print(f"削減率: {(1 - params_tied/params_normal)*100:.2f}%")

## 6.5.6 まとめと評価
本節の改善により、PTB データセットに対するテストデータのパープレキシティは、改善前の約 136 から約 75 前後まで大幅に向上する  

- 多層化で表現力を高め、
- Dropout で汎用性を向上させ、
- 重み共有でパラメータを効率化

現代の最先端研究 (Transformer や Attention の活用など) も、これら多層化や Dropout ベースの正則化といった共通の基盤の上に成り立っている  

# 6.6 まとめ

本章では、シンプルな RNN が抱えていた課題を解決し、長期の依存関係を学習可能にする「ゲート付き RNN」(特に LSTM) について詳しく学んだ  

### 本章の振り返り
- 勾配消失と勾配爆発：  
    シンプルな RNN の学習では、BPTT において勾配が指数的に変化する問題があった  

- 勾配爆発への対策：  
    勾配のノルムをしきい値で抑える勾配クリッピングが有効  

- 勾配消失への対策：  
    ゲートという仕組みを持つ LSTM や GRU などのレイヤが有効  

- LSTM の 3 つのゲート：  
    - forget ゲート：記憶セルから何を忘れるかを決める  

    - input ゲート：記憶セルに何を新しく追加するかを決める  

    - output ゲート：次時刻の隠れ状態として何を出力するかを決める  

- RNNLM の改善：  
    言語モデルの精度向上のため、多層化、Dropout (深さ方向)、重み共有 (Embedding と Affine 間) といったテクニックを導入し、パープレキシティを大幅に改善した  

### 第 6 章の技術マップ
本章で構築した改善技術のつながりを視覚的に整理する  

```mermaid
graph TD
    A[シンプルなRNN] -->|課題: 長期依存関係の学習困難| B{勾配問題}
    B -->|解決: 勾配爆発| C[勾配クリッピング]
    B -->|解決: 勾配消失| D[ゲート付きRNN: LSTM / GRU]
    
    D --> E[RNNLMの構築]
    E --> F{さらなる精度向上策}
    F --> G[レイヤの多層化]
    F --> H[深さ方向へのDropout]
    F --> I[重み共有]
    
    G & H & I --> J[高性能な言語モデル]
    J -->|次章のテーマへ| K[文章生成 / seq2seq]
```

# 次章への橋渡し：学習したモデルで「言葉」を紡ぐ

本章では、大量のテキストデータから単語の並びのパターンを学習した「高品質な言語モデル」を作成した。次章 (第 7 章) では、このモデルを使って、コンピュータに文章を生成させたり、ある時系列を別の時系列に変換する seq2seq という仕組みを学ぶ  

### 第 7 章の内容
1.  AI に文章を書かせる：  
    学習した重みを用いて、ある単語に続く単語を確率的に選択し続けることで、新しい文章を自動生成する  

2.  seq2seq (Sequence to Sequence)：  
    「Encoder」で入力をエンコードし、「Decoder」で別の言語などにデコードする、機械翻訳やチャットボットの基盤技術を実装する  

3.  足し算を学習する：  
    「57+5」という文字列から「62」という答えを導き出すような、アルゴリズムの学習に挑戦する  